In [1]:
import torch
from d2l import torch as d2l
import torch.nn as nn

In [2]:
import numpy
import math
import time

In [3]:
class benchmark:
    def __init__(self,description):
        self.name=description
        return
    def __enter__(self):
        self.start_time=time.time()
    def __exit__(self,*args):
        print(f"{self.name}:duration{time.time()-self.start_time:.3f}s")


net=nn.Sequential(nn.Linear(10,12),nn.ReLU(),
                  nn.Linear(12,8),nn.ReLU(),
                  nn.Linear(8,1),nn.ReLU()).to(torch.device('cuda:2'))

net0=torch.jit.script(net)

#编译后运算
with benchmark("no script"):
    for i in range(10000):
        x=torch.rand(1024,10).to(torch.device('cuda:2'))
        y=net(x)

with benchmark("script"):
    for i in range(10000):
        x=torch.rand(1024,10).to(torch.device('cuda:2'))
        y=net0(x)

/nfs3/wangfengyu/conda_envs/mytorch/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


no script:duration2.477s
script:duration2.187s


In [4]:
device = torch.device('cuda:0')

#设备预热：GPU在首次调用时计算time可能会有偏差。
for i in range(100):
    x=torch.randn([1024,1024]).to(device)
    y=torch.normal(0,1,[1024,1024]).to(device)
    torch.mm(x,y)

torch.cuda.synchronize(device)   #CUDA默认CPU与GPU之间是并行的，CPU发送完GPU指令可以继续执行计算，synchronize会要求等待该设备的计算完成后才继续进行下一步cpu的执行

mm=torch.zeros([1024,1024]).to(device)
#CPU派发指令
with benchmark("cpu send"):
    for i in range(100):
        a = torch.randn(size=(1000, 1000), device=device)
        b=torch.mm(x,y)

mm=torch.zeros([1024,1024]).to(device)
#GPU运算
with benchmark("gpu finish"):
    for i in range(100):
        a = torch.randn(size=(1000, 1000), device=device)
        b=torch.mm(x,y)
    torch.cuda.synchronize(device)    #如果不加synchronize，测出的实际是CPU下发完指令的时间。

mm=torch.zeros([1024,1024]).to(device)

#GPU运算
with benchmark("gpu rely"):
    for i in range(100):
        a = torch.randn(size=(1000, 1000), device=device)
        b=torch.mm(x,y)
            #如果GPU的执行结果需要被使用，在mm+=处，cpu会等待gpu完成。

with benchmark("numpy"):
    for i in range(10):
        x=numpy.random.normal(size=(1024,1024))
        y=numpy.random.normal(size=(1024,1024))
        numpy.dot(x,y)

cpu send:duration0.009s
gpu finish:duration0.030s
gpu rely:duration0.004s
numpy:duration0.932s


In [5]:
torch.cuda.synchronize(torch.device('cuda:1'))  
torch.cuda.synchronize(torch.device('cuda:2'))  

devices = d2l.try_all_gpus() 
def run(x):  
    return [x.mm(x) for _ in range(50)] 
x_gpu1 = torch.rand(size=(4000, 4000), device=devices[0]) 
x_gpu2 = torch.rand(size=(4000, 4000), device=devices[1])

run(x_gpu1) 
run(x_gpu2) # 预热设备 
torch.cuda.synchronize(devices[0]) 
torch.cuda.synchronize(devices[1]) 
with d2l.Benchmark('GPU1 time'):
  run(x_gpu1) 
  torch.cuda.synchronize(devices[0])
  
with d2l.Benchmark('GPU2 time'):
    run(x_gpu2)
    torch.cuda.synchronize(devices[1])

with d2l.Benchmark('GPU1 & GPU2'):
      run(x_gpu1)
      run(x_gpu2)
      torch.cuda.synchronize()

GPU1 time: 0.6288 sec
GPU2 time: 0.6847 sec
GPU1 & GPU2: 0.6791 sec


In [ ]:
cpu=torch.device('cpu')
x_gpu = torch.rand(size=(4000, 4000), device=devices[0]) 

def copy_to_device(target_device,data,non_blocking=False):
    return [y.to(target_device,non_blocking=non_blocking) for y in data]

with d2l.Benchmark('run on GPU'):
    t=run(x_gpu)
    torch.cuda.synchronize(devices[0])

with d2l.Benchmark('copy back'):
    z = run(x_gpu)
    z=copy_to_device(cpu,z)
    torch.cuda.synchronize(devices[0])

with d2l.Benchmark('copy back'):
    z = run(x_gpu)
    z=copy_to_device(cpu,z,True)
    torch.cuda.synchronize(devices[0])     #在系统能自动实现在前者GPU执行的同时，把已经计算好的部分数据提前开始传输

with d2l.Benchmark('copy back'):
    z = run(x_gpu)
    z=copy_to_device(device,z,True)
    torch.cuda.synchronize(devices[0])     #GPU间的传输要比CPU快很多。

run on GPU: 0.6693 sec
copy back: 3.1920 sec
copy back: 0.7255 sec
copy back: 0.3180 sec
